# Solar filament segmentation

Attach the official competition data and enable GPU and Internet. Mode `all` trains, calibrates, and exports in one run. Mode `submit` uses an attached best.pt, recalibrates, and exports. See `KAGGLE_UPLOAD_GUIDE.md` for upload instructions.

In [ ]:
RUN_MODE = "submit"
EXPERIMENT = "r34_bcedice_v2_seed0_hann_tta"
DATA_ROOT = ""
CALIBRATION_PATH = ""
CHECKPOINTS = []
CHECKPOINT_SEEDS = [0]

MODEL = "smp"
ENCODER = "resnet34"
BASE = 16
SPLIT_SEED = 0
TTA = True
TILE_SIZE = 512
TILE_OVERLAP = 128
BLEND = "hann"
TUNE_OBSERVATIONS = 24
USE_TUNED_POSTPROCESSING = True
RUN_COMPRESSION = False
EXPECTED_TEST_IMAGES = 180


TRAIN_SEEDS = [0]
EPOCHS = 60
SAMPLES_PER_EPOCH = 3584
BATCH_SIZE = 4
CROP_SIZE = 512
LEARNING_RATE = 3e-4
CLDICE_WEIGHT = 0.0
SAMPLING = "annotation"
WORKERS = 2
INIT_CHECKPOINT = ""
RESUME_CHECKPOINT = ""

In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path

if RUN_MODE not in {"all", "submit", "train", "calibrate", "infer"}:
    raise ValueError("RUN_MODE must be all, submit, train, calibrate, or infer")
if RUN_MODE == "all" and CHECKPOINTS:
    raise ValueError("Use RUN_MODE='submit' for existing CHECKPOINTS; clear CHECKPOINTS for fresh all-mode training.")
if RUN_MODE in {"all", "submit"} and CALIBRATION_PATH:
    raise ValueError("Leave CALIBRATION_PATH blank in automatic modes; calibration is generated in this run.")
if not EXPERIMENT or Path(EXPERIMENT).name != EXPERIMENT:
    raise ValueError("EXPERIMENT must be one folder name")
if importlib.util.find_spec("torch") is None:
    raise RuntimeError("Select the project's .venv Jupyter kernel locally, or a Kaggle GPU runtime with PyTorch.")
dependencies = {
    "numpy": "numpy>=1.26,<3", "pandas": "pandas>=2,<3",
    "cv2": "opencv-python-headless>=4.8,<5", "scipy": "scipy>=1.11,<2",
    "segmentation_models_pytorch": "segmentation-models-pytorch>=0.4,<1",
    "pycocotools": "pycocotools>=2.0.7,<3", "skimage": "scikit-image>=0.22,<1",
    "matplotlib": "matplotlib>=3.7,<4",
}
missing = [requirement for name, requirement in dependencies.items()
           if importlib.util.find_spec(name) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

import json
import shutil
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU in Kaggle settings, or select your CUDA-enabled local environment.")
print("GPU:", torch.cuda.get_device_name(0), "| PyTorch:", torch.__version__)
ORIGINAL_CWD = Path.cwd().resolve()
ON_KAGGLE = Path("/kaggle/input").is_dir()
WORK = Path("/kaggle/working") if ON_KAGGLE else ORIGINAL_CWD / "runs" / "notebook_working"
CODE_DIR = WORK / "filament_code"
EXP_DIR = WORK / EXPERIMENT
CODE_DIR.mkdir(parents=True, exist_ok=True)
EXP_DIR.mkdir(parents=True, exist_ok=True)
print("Experiment outputs:", EXP_DIR)

In [ ]:
EMBEDDED_SOURCES = {'checkpoint_io.py': 'import hashlib\nimport os\nfrom pathlib import Path\nimport tempfile\nimport zipfile\n\nimport torch\n\n\ndef is_unpacked_checkpoint(path):\n    path = Path(path)\n    return (path.is_dir() and (path / "data.pkl").is_file()\n            and (path / "version").is_file() and (path / "data").is_dir())\n\n\ndef find_checkpoints(roots):\n    found = set()\n    for root in map(Path, roots):\n        candidates = root.rglob("best.pt") if root.is_dir() else [root]\n        for candidate in candidates:\n            if candidate.name == "best.pt" and (candidate.is_file() or is_unpacked_checkpoint(candidate)):\n                found.add(candidate.resolve())\n    return sorted(found)\n\n\ndef _content_hash(members):\n    digest = hashlib.sha256()\n    for name, path in members:\n        encoded = name.encode("utf-8")\n        digest.update(len(encoded).to_bytes(8, "little"))\n        digest.update(encoded)\n        digest.update(path.stat().st_size.to_bytes(8, "little"))\n        with path.open("rb") as handle:\n            for part in iter(lambda: handle.read(1024 * 1024), b""):\n                digest.update(part)\n    return digest.hexdigest()\n\n\ndef _validate(path):\n    torch.load(path, map_location="cpu", weights_only=True)\n\n\ndef prepare_checkpoint(path, cache_dir):\n    source = Path(path).expanduser().resolve()\n    if source.suffix.lower() == ".zip":\n        raise ValueError("Extract only the downloaded outer ZIP and select best.pt inside it. "\n                         "Keep the best.pt checkpoint itself intact.")\n    if source.is_file():\n        return source\n    if not source.exists():\n        raise FileNotFoundError(f"Checkpoint does not exist in this session: {source}")\n    if not is_unpacked_checkpoint(source):\n        raise ValueError(f"Checkpoint path is a folder, not an intact best.pt or recognized unpacked "\n                         f"PyTorch checkpoint: {source}. Expected data.pkl, version, and data/.")\n\n    entries = sorted(source.rglob("*"))\n    if any(entry.is_symlink() for entry in entries):\n        raise ValueError(f"Unpacked checkpoint contains symbolic links: {source}")\n    members = [(entry.relative_to(source).as_posix(), entry) for entry in entries if entry.is_file()]\n    digest = _content_hash(members)\n    destination = Path(cache_dir).expanduser().resolve() / digest / "best.pt"\n    if destination.is_file():\n        try:\n            _validate(destination)\n            return destination\n        except Exception:\n            pass\n\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    with tempfile.NamedTemporaryFile(dir=destination.parent, suffix=".tmp", delete=False) as handle:\n        temporary = Path(handle.name)\n    try:\n        with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_STORED) as archive:\n            for name, member in members:\n                archive.write(member, f"archive/{name}")\n        try:\n            _validate(temporary)\n        except Exception as error:\n            raise ValueError(f"Could not reconstruct valid PyTorch weights from {source}. "\n                             "Check that all files from the original checkpoint were preserved.") from error\n        os.replace(temporary, destination)\n    finally:\n        temporary.unlink(missing_ok=True)\n    return destination\n', 'data.py': '\n\n\nimport cv2\nimport numpy as np\nimport pandas as pd\n\n\ndef load_image(path):\n    path = str(path)\n    if path.endswith(".npy"):\n        img = np.load(path).astype(np.float32)\n    elif path.endswith((".fits", ".fts", ".fit")):\n        from astropy.io import fits\n        img = fits.getdata(path).astype(np.float32)\n    else:\n        raw = cv2.imread(path, cv2.IMREAD_UNCHANGED)\n        if raw is None:\n            raise FileNotFoundError(f"Could not load image: {path}")\n        img = raw.astype(np.float32)\n        if img.ndim == 3:\n            img = img.mean(-1)\n    img = np.nan_to_num(img)\n    lo, hi = np.percentile(img, [1, 99.5])\n    return np.clip((img - lo) / (hi - lo + 1e-6), 0, 1).astype(np.float32)\n\n\ndef load_instance_mask(path):\n    path = str(path)\n    m = np.load(path) if path.endswith(".npy") else cv2.imread(path, cv2.IMREAD_UNCHANGED)\n    if m is None:\n        raise FileNotFoundError(f"Could not load instance mask: {path}")\n    if m.ndim == 3:\n        m = m[..., 0]\n    if m.max() <= 1:\n        _, m = cv2.connectedComponents((m > 0).astype(np.uint8), connectivity=8)\n    return m.astype(np.int32)\n\n\ndef _polys_to_mask(polys, h, w):\n    from pycocotools import mask as mu\n    if not polys:\n        return np.zeros((h, w), dtype=np.uint8)\n    rles = mu.frPyObjects(polys, h, w)\n    return mu.decode(mu.merge(rles)).astype(np.uint8)\n\n\ndef build_manifest(root, cache_dir="masks", out_csv="manifest.csv"):\n    import json, os\n    from collections import defaultdict\n    from pathlib import Path\n    root = Path(root).resolve()\n    os.makedirs(cache_dir, exist_ok=True)\n    json_files = sorted((root / "train").glob("*.json"))\n    if len(json_files) != 1:\n        raise ValueError(f"Expected one training COCO JSON under {root / \'train\'}, found: {json_files}")\n    with open(json_files[0], encoding="utf-8") as handle:\n        coco = json.load(handle)\n    anns = defaultdict(list)\n    for a in coco["annotations"]:\n        anns[str(a["image_id"])].append(a)\n\n    rows = []\n    for im in coco["images"]:\n        h, w = im["height"], im["width"]\n        inst = np.zeros((h, w), np.uint16)\n        for k, a in enumerate(anns[str(im["id"])], start=1):\n            inst[_polys_to_mask(a["segmentation"], h, w) > 0] = k\n        mp = Path(cache_dir) / f"{im[\'id\']}.png"\n        if not cv2.imwrite(str(mp), inst):\n            raise OSError(f"Unable to write instance mask: {mp}")\n        stem = Path(im["file_name"]).stem\n        image_path = root / "train" / "train_images" / im["file_name"]\n        if not image_path.exists():\n            raise FileNotFoundError(f"Training image missing: {image_path}")\n        rows.append(dict(image_path=str(image_path),\n                         mask_path=str(mp), image_id=str(im["id"]), obs_id=stem,\n                         time=stem[:14], instrument=stem[14:], n_filaments=len(anns[str(im["id"])]),\n                         split="train"))\n    test_dir = root / "test" / "test_images"\n    if not test_dir.is_dir():\n        raise FileNotFoundError(f"Test image directory missing: {test_dir}")\n    for p in sorted(test_dir.iterdir()):\n        if not p.is_file() or p.suffix.lower() not in {".jpg", ".jpeg", ".png"}:\n            continue\n        stem = p.stem\n        rows.append(dict(image_path=str(p), mask_path="", image_id=stem, obs_id=stem,\n                         time=stem[:14], instrument=stem[14:], n_filaments=-1, split="test"))\n    df = pd.DataFrame(rows)\n    Path(out_csv).parent.mkdir(parents=True, exist_ok=True)\n    df.to_csv(out_csv, index=False)\n    print(df.groupby("split").size().to_dict(), "| instruments:", df.instrument.value_counts().to_dict())\n    return df\n\n\ndef time_block_split(df, block=20, val_every=5, buffer=2, seed=0):\n    obs = (df.drop_duplicates("obs_id").sort_values("time")["obs_id"].tolist())\n    n_blocks = int(np.ceil(len(obs) / block))\n    offset = np.random.default_rng(seed).integers(0, val_every)\n    val, drop = set(), set()\n    for b in range(n_blocks):\n        lo, hi = b * block, min((b + 1) * block, len(obs))\n        if (b + offset) % val_every == 0:\n            val.update(obs[lo:hi])\n            drop.update(obs[max(0, lo - buffer):lo]); drop.update(obs[hi:hi + buffer])\n    train = [o for o in obs if o not in val and o not in drop]\n    return df[df.obs_id.isin(train)].reset_index(drop=True), \\\n           df[df.obs_id.isin(val)].reset_index(drop=True)\n\n\ndef pad_to_multiple(img, m=32, value=0):\n    h, w = img.shape[-2:]\n    ph, pw = (-h) % m, (-w) % m\n    pad = [(0, 0)] * (img.ndim - 2) + [(0, ph), (0, pw)]\n    return np.pad(img, pad, constant_values=value), (h, w)\n\n\ndef random_crop(img, mask, size, pos_prob=0.7, rng=np.random):\n    h, w = img.shape\n    size = min(size, h, w)\n    ys, xs = np.nonzero(mask) if (mask.any() and rng.rand() < pos_prob) else (None, None)\n    if ys is not None:\n        k = rng.randint(len(ys))\n        cy, cx = ys[k], xs[k]\n        y0 = np.clip(cy - rng.randint(size), 0, h - size)\n        x0 = np.clip(cx - rng.randint(size), 0, w - size)\n    else:\n        y0, x0 = rng.randint(0, h - size + 1), rng.randint(0, w - size + 1)\n    return img[y0:y0 + size, x0:x0 + size], mask[y0:y0 + size, x0:x0 + size]\n\n\ndef augment(img, mask, rng=np.random):\n    if rng.rand() < 0.5: img, mask = img[:, ::-1], mask[:, ::-1]\n    if rng.rand() < 0.5: img, mask = img[::-1], mask[::-1]\n    k = rng.randint(4); img, mask = np.rot90(img, k), np.rot90(mask, k)\n    img = np.ascontiguousarray(img); mask = np.ascontiguousarray(mask)\n\n    if rng.rand() < 0.8:\n        img = np.clip(img * rng.uniform(0.8, 1.2) + rng.uniform(-0.1, 0.1), 0, 1)\n        img = img ** rng.uniform(0.8, 1.25)\n    if rng.rand() < 0.3:\n        s = rng.uniform(0.5, 1.5)\n        img = cv2.GaussianBlur(img, (0, 0), s)\n    if rng.rand() < 0.3:\n        img = np.clip(img + rng.normal(0, rng.uniform(0.005, 0.03), img.shape), 0, 1)\n    return img.astype(np.float32), mask\n\n\ndef rle_encode(mask):\n    from pycocotools import mask as mu\n    binary = np.asfortranarray(np.asarray(mask, dtype=np.uint8))\n    if binary.ndim != 2:\n        raise ValueError(f"RLE mask must be HxW; got shape={binary.shape}")\n    return mu.encode(binary)["counts"].decode("ascii")\n\n\ndef rle_counts(encoded):\n    if not isinstance(encoded, str) or not encoded:\n        raise ValueError("Compressed RLE must be a nonempty string")\n    counts = []\n    position = 0\n    while position < len(encoded):\n        value, shift = 0, 0\n        while True:\n            if position == len(encoded):\n                raise ValueError("Truncated compressed RLE count")\n            code = ord(encoded[position]) - 48\n            position += 1\n            if not 0 <= code <= 63:\n                raise ValueError("Invalid character in compressed RLE")\n            value |= (code & 31) << shift\n            shift += 5\n            if not code & 32:\n                if code & 16:\n                    value -= 1 << shift\n                break\n            if shift > 60:\n                raise ValueError("Compressed RLE count is too large")\n        if len(counts) > 2:\n            value += counts[-2]\n        if value < 0:\n            raise ValueError("Compressed RLE contains a negative run length")\n        counts.append(value)\n    return counts\n\n\ndef rle_decode(s, shape):\n    from pycocotools import mask as mu\n    if not isinstance(s, str) or not s:\n        return np.zeros(shape, dtype=np.uint8)\n    counts = rle_counts(s)\n    pixels = int(np.prod(shape))\n    if sum(counts) != pixels:\n        raise ValueError(f"RLE runs cover {sum(counts)} pixels; expected {pixels}")\n    return mu.decode({"counts": s.encode("ascii"), "size": list(shape)}).astype(np.uint8)\n\n\ndef _selftest():\n    m = np.zeros((50, 60), np.uint8); m[5:20, 10:30] = 1; m[30:35, 40:58] = 1\n    assert (rle_decode(rle_encode(m), m.shape) == m).all(); print("RLE round-trip OK")\n\n    df = pd.DataFrame({"obs_id": [f"o{i}" for i in range(200) for _ in range(2)],\n                       "time": [i for i in range(200) for _ in range(2)]})\n    tr, va = time_block_split(df)\n    print(len(tr), len(va), "overlap of obs ids:", set(tr.obs_id) & set(va.obs_id))\n\n    img = np.random.rand(300, 300).astype(np.float32); mk = np.zeros((300, 300), np.int32); mk[100:110, 50:250] = 1\n    ci, cm = random_crop(img, mk, 128); ai, am = augment(ci, cm); print("crop/aug shapes", ai.shape, am.shape)\n\n\nif __name__ == "__main__":\n    import sys\n    if len(sys.argv) > 1 and sys.argv[1] == "build":\n        import argparse\n        ap = argparse.ArgumentParser(); ap.add_argument("build")\n        ap.add_argument("--root", required=True); ap.add_argument("--cache", default="masks")\n        ap.add_argument("--out", default="manifest.csv")\n        a = ap.parse_args(); build_manifest(a.root, a.cache, a.out)\n    else:\n        _selftest()\n', 'models.py': 'import time\nimport numpy as np\nimport torch\nimport torch.nn as nn\n\nfrom data import pad_to_multiple\n\n\ndef build_unet(encoder="resnet34", weights="imagenet"):\n    import segmentation_models_pytorch as smp\n    return smp.Unet(encoder_name=encoder, encoder_weights=weights, in_channels=1, classes=1)\n\n\nclass ConvBlock(nn.Module):\n    def __init__(self, cin, cout):\n        super().__init__()\n        self.net = nn.Sequential(\n            nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),\n            nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))\n    def forward(self, x):\n        return self.net(x)\n\n\nclass LightUNet(nn.Module):\n    def __init__(self, base=16, depth=4):\n        super().__init__()\n        ch = [base * 2 ** i for i in range(depth + 1)]\n        self.enc = nn.ModuleList([ConvBlock(1 if i == 0 else ch[i - 1], ch[i]) for i in range(depth + 1)])\n        self.pool = nn.MaxPool2d(2)\n        self.up = nn.ModuleList([nn.ConvTranspose2d(ch[i + 1], ch[i], 2, stride=2) for i in range(depth)])\n        self.dec = nn.ModuleList([ConvBlock(ch[i] * 2, ch[i]) for i in range(depth)])\n        self.head = nn.Conv2d(ch[0], 1, 1)\n\n    def forward(self, x):\n        skips = []\n        for i, blk in enumerate(self.enc):\n            x = blk(x)\n            if i < len(self.enc) - 1:\n                skips.append(x); x = self.pool(x)\n        for i in reversed(range(len(self.dec))):\n            x = self.up[i](x)\n            x = self.dec[i](torch.cat([x, skips[i]], 1))\n        return self.head(x)\n\n\ndef build_model(kind="smp", encoder="resnet34", weights="imagenet", base=16):\n    return build_unet(encoder, weights) if kind == "smp" else LightUNet(base=base)\n\n\ndef count_params(model):\n    return sum(p.numel() for p in model.parameters())\n\n\ndef model_size_mb(model):\n    n = sum(t.numel() * t.element_size() for t in list(model.parameters()) + list(model.buffers()))\n    return n / 1024 ** 2\n\n\n@torch.no_grad()\ndef predict_prob(model, img, device, tta=False, half=False, tile_size=512, overlap=64,\n                 blend="uniform"):\n\n    model.eval()\n    if tile_size <= 0 or overlap < 0 or overlap >= tile_size:\n        raise ValueError("Require tile_size > 0 and 0 <= overlap < tile_size")\n    if blend not in {"uniform", "hann"}:\n        raise ValueError("blend must be \'uniform\' or \'hann\'")\n    img = np.asarray(img)\n    if img.ndim != 2 or not all(img.shape):\n        raise ValueError("Expected a nonempty HxW image")\n    h, w = img.shape\n    stride = tile_size - overlap\n\n    def starts(n):\n        if n <= tile_size:\n            return [0]\n        arr = list(range(0, n - tile_size + 1, stride))\n        if arr[-1] != n - tile_size:\n            arr.append(n - tile_size)\n        return arr\n\n    acc = np.zeros((h, w), dtype=np.float32)\n    count = np.zeros((h, w), dtype=np.float32)\n    weights = {}\n    for y0 in starts(h):\n        for x0 in starts(w):\n            tile = np.ascontiguousarray(img[y0:y0+tile_size, x0:x0+tile_size])\n            padded, (th, tw) = pad_to_multiple(tile, 32)\n            inp = torch.from_numpy(np.ascontiguousarray(padded))[None, None].to(device)\n            if half:\n                inp = inp.half()\n            def fwd(x):\n                with torch.autocast(device_type=device.type, enabled=(device.type == "cuda" and not half)):\n                    return torch.sigmoid(model(x).float())\n            outs = [fwd(inp)]\n            if tta:\n                outs += [fwd(inp.flip((-1,))).flip((-1,)),\n                         fwd(inp.flip((-2,))).flip((-2,)),\n                         fwd(inp.flip((-1, -2))).flip((-1, -2))]\n            patch = torch.stack(outs).mean(0)[0, 0, :th, :tw].cpu().numpy()\n            if (th, tw) not in weights:\n                if blend == "hann":\n\n\n                    weights[th, tw] = np.maximum(\n                        np.outer(np.hanning(th), np.hanning(tw)), 1e-3\n                    ).astype(np.float32)\n                else:\n                    weights[th, tw] = np.ones((th, tw), dtype=np.float32)\n            weight = weights[th, tw]\n            acc[y0:y0+th, x0:x0+tw] += patch * weight\n            count[y0:y0+th, x0:x0+tw] += weight\n    return acc / np.maximum(count, np.finfo(np.float32).tiny)\n\n\n@torch.no_grad()\ndef time_inference(model, device, shape=(1024, 1024), n=20, half=False):\n    model.eval()\n    x = torch.rand(1, 1, *shape, device=device)\n    x = x.half() if half else x\n    for _ in range(3):\n        model(x)\n    if device.type == "cuda":\n        torch.cuda.synchronize()\n    t = time.perf_counter()\n    for _ in range(n):\n        model(x)\n    if device.type == "cuda":\n        torch.cuda.synchronize()\n    return (time.perf_counter() - t) / n\n', 'losses.py': '\nimport torch\nimport torch.nn.functional as F\n\n\ndef dice_loss(prob, target, eps=1.0):\n    dims = (1, 2, 3)\n    inter = (prob * target).sum(dims)\n    return 1 - ((2 * inter + eps) / (prob.sum(dims) + target.sum(dims) + eps)).mean()\n\n\ndef _erode(x):\n    return torch.min(-F.max_pool2d(-x, (3, 1), 1, (1, 0)), -F.max_pool2d(-x, (1, 3), 1, (0, 1)))\n\ndef _dilate(x):\n    return F.max_pool2d(x, 3, 1, 1)\n\ndef soft_skeleton(x, iters=10):\n    skel = F.relu(x - _dilate(_erode(x)))\n    for _ in range(iters):\n        x = _erode(x)\n        delta = F.relu(x - _dilate(_erode(x)))\n        skel = skel + F.relu(delta - skel * delta)\n    return skel\n\ndef cl_dice_loss(prob, target, iters=10, eps=1.0):\n    sp, st = soft_skeleton(prob, iters), soft_skeleton(target, iters)\n    t_prec = ((sp * target).sum((1, 2, 3)) + eps) / (sp.sum((1, 2, 3)) + eps)\n    t_sens = ((st * prob).sum((1, 2, 3)) + eps) / (st.sum((1, 2, 3)) + eps)\n    return (1 - 2 * t_prec * t_sens / (t_prec + t_sens)).mean()\n\n\ndef seg_loss(logits, target, w_bce=1.0, w_dice=1.0, w_cldice=0.0):\n    loss = w_bce * F.binary_cross_entropy_with_logits(logits, target)\n    prob = torch.sigmoid(logits.float())\n    if w_dice:\n        loss = loss + w_dice * dice_loss(prob, target)\n    if w_cldice:\n        loss = loss + w_cldice * cl_dice_loss(prob, target)\n    return loss\n\n\ndef distill_loss(student_logits, teacher_logits, T=2.0):\n    soft_t = torch.sigmoid(teacher_logits / T)\n    return F.binary_cross_entropy_with_logits(student_logits / T, soft_t) * (T * T)\n', 'pq_metric.py': 'import numpy as np\n\n\ndef match_instances(gt, pred, iou_thr=0.5):\n\n    gt, pred = np.asarray(gt), np.asarray(pred)\n    if gt.shape != pred.shape or gt.ndim != 2:\n        raise ValueError("Ground truth and prediction must have the same HxW shape")\n    if not 0.5 <= iou_thr <= 1:\n        raise ValueError("Exclusive-map PQ requires 0.5 <= iou_thr <= 1")\n    if gt.dtype.kind not in "biu" or pred.dtype.kind not in "biu":\n        raise ValueError("Instance maps must contain integer labels")\n    g, p = gt.ravel(), pred.ravel()\n    if (g.size and g.min() < 0) or (p.size and p.min() < 0):\n        raise ValueError("Instance labels must be nonnegative")\n    gm, pm = int(g.max(initial=0)), int(p.max(initial=0))\n    both = (g > 0) & (p > 0)\n\n\n    if (gm + 1) * (pm + 1) <= 1_000_000:\n        g = g.astype(np.int64, copy=False)\n        p = p.astype(np.int64, copy=False)\n        ga = np.bincount(g, minlength=gm + 1)\n        pa = np.bincount(p, minlength=pm + 1)\n        ng, npred = np.count_nonzero(ga[1:]), np.count_nonzero(pa[1:])\n        base = pm + 1\n        counts = np.bincount(g[both] * base + p[both])\n        codes = np.flatnonzero(counts)\n        gi, pi = codes // base, codes % base\n        inter = counts[codes]\n        g_inter_area, p_inter_area = ga[gi], pa[pi]\n    else:\n\n\n        g_ids, g_counts = np.unique(g[g > 0], return_counts=True)\n        p_ids, p_counts = np.unique(p[p > 0], return_counts=True)\n        ng, npred = len(g_ids), len(p_ids)\n\n        compact_g = np.searchsorted(g_ids, g[both])\n        compact_p = np.searchsorted(p_ids, p[both])\n        pairs, inter = np.unique(np.column_stack((compact_g, compact_p)),\n                                 axis=0, return_counts=True)\n        gi, pi = pairs[:, 0], pairs[:, 1]\n        g_inter_area, p_inter_area = g_counts[gi], p_counts[pi]\n\n    ious = inter / (g_inter_area + p_inter_area - inter)\n    matched = ious > iou_thr\n    tp = int(matched.sum())\n\n    _, split_counts = np.unique(gi[inter / p_inter_area >= 0.1], return_counts=True)\n    _, merge_counts = np.unique(pi[inter / g_inter_area >= 0.1], return_counts=True)\n    return (tp, int(npred - tp), int(ng - tp), float(ious[matched].sum()),\n            int((split_counts > 1).sum()), int((merge_counts > 1).sum()))\n\n\nclass PQAccumulator:\n    def __init__(self):\n        self.tp = self.fp = self.fn = 0\n        self.iou = 0.0\n        self.split = self.merge = 0\n\n    def update(self, gt, pred):\n        tp, fp, fn, iou, s, m = match_instances(gt, pred)\n        self.tp += tp; self.fp += fp; self.fn += fn\n        self.iou += iou; self.split += s; self.merge += m\n\n    def compute(self):\n        denom = self.tp + 0.5 * self.fp + 0.5 * self.fn\n        pq = self.iou / denom if denom > 0 else 0.0\n        sq = self.iou / self.tp if self.tp > 0 else 0.0\n        rq = self.tp / denom if denom > 0 else 0.0\n        return dict(PQ=pq, SQ=sq, RQ=rq, TP=self.tp, FP=self.fp, FN=self.fn,\n                    split=self.split, merge=self.merge)\n\n\nif __name__ == "__main__":\n\n    gt = np.zeros((100, 100), int); gt[10:30, 10:60] = 1; gt[60:80, 10:60] = 2\n    acc = PQAccumulator(); acc.update(gt, gt.copy())\n    print("perfect  :", acc.compute()["PQ"])\n\n    frag = gt.copy(); frag[10:30, 35:60] = 3\n    acc = PQAccumulator(); acc.update(gt, frag)\n    print("fragment :", acc.compute())\n\n    merged = np.where(gt > 0, 1, 0)\n    acc = PQAccumulator(); acc.update(gt, merged)\n    print("merged   :", acc.compute())\n', 'postprocess.py': 'import itertools\nimport cv2\nimport numpy as np\nfrom skimage.filters import apply_hysteresis_threshold\nfrom skimage.feature import peak_local_max\nfrom skimage.segmentation import watershed as sk_watershed\nfrom scipy import ndimage as ndi\n\nfrom pq_metric import PQAccumulator\n\nDEFAULTS = dict(low=0.35, high=0.6, close_px=3, min_area=30, watershed=False, ws_min_dist=15)\n\n\ndef prob_to_instances(prob, low=0.35, high=0.6, close_px=3, min_area=30,\n                      watershed=False, ws_min_dist=15):\n\n    mask = apply_hysteresis_threshold(prob, low, high).astype(np.uint8)\n\n\n    if close_px > 0:\n        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * close_px + 1, 2 * close_px + 1))\n        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k)\n\n\n    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)\n    keep = np.zeros(n, dtype=bool)\n    keep[1:] = stats[1:, cv2.CC_STAT_AREA] >= min_area\n    mask = keep[labels]\n\n    if watershed and mask.any():\n        dist = ndi.distance_transform_edt(mask)\n        pts = peak_local_max(dist, min_distance=ws_min_dist, labels=mask.astype(int),\n                             exclude_border=False)\n        markers = np.zeros(mask.shape, np.int32)\n        markers[tuple(pts.T)] = np.arange(1, len(pts) + 1)\n        return sk_watershed(-dist, markers, mask=mask).astype(np.int32)\n\n    n, labels = cv2.connectedComponents(mask.astype(np.uint8), connectivity=8)\n    return labels.astype(np.int32)\n\n\ndef evaluate(probs, gts, **params):\n    if len(probs) != len(gts) or not len(probs):\n        raise ValueError("Need equal, nonempty probability and ground-truth lists")\n    acc = PQAccumulator()\n    for p, g in zip(probs, gts):\n        acc.update(g, prob_to_instances(p, **params))\n    return acc.compute()\n\n\ndef tune(probs, gts, grid=None, verbose=True):\n\n\n    if grid is None:\n        candidates = dict(low=[0.2, 0.35, 0.45, 0.6],\n                          high=[0.5, 0.6, 0.7, 0.85],\n                          close_px=[0, 3, 6, 9],\n                          min_area=[30, 100, 300, 1000])\n        best, best_pq, scores = dict(DEFAULTS), -1.0, {}\n\n        def score(params):\n            key = tuple(sorted(params.items()))\n            if key not in scores:\n                result = evaluate(probs, gts, **params)\n                scores[key] = result["PQ"]\n                if verbose:\n                    print(params, f"PQ={result[\'PQ\']:.4f}", flush=True)\n            return scores[key]\n\n\n        for start in [dict(DEFAULTS), {**DEFAULTS, "low": 0.45,\n                                       "high": 0.7, "close_px": 6, "min_area": 100}]:\n            current, value = start, score(start)\n            for _ in range(2):\n                changed = False\n                for name, values in candidates.items():\n                    for candidate in values:\n                        params = {**current, name: candidate}\n                        if params["low"] > params["high"]:\n                            continue\n                        pq = score(params)\n                        if pq > value:\n                            current, value, changed = params, pq, True\n                if not changed:\n                    break\n            if value > best_pq:\n                best, best_pq = current, value\n        return best, best_pq\n    keys = list(grid)\n    best, best_pq = None, -1\n    for combo in itertools.product(*[grid[k] for k in keys]):\n        params = dict(zip(keys, combo))\n        if params["low"] > params["high"]:\n            continue\n        r = evaluate(probs, gts, **params)\n        if verbose:\n            print(params, f"PQ={r[\'PQ\']:.4f} split={r[\'split\']} merge={r[\'merge\']}")\n        if r["PQ"] > best_pq:\n            best, best_pq = params, r["PQ"]\n    return best, best_pq\n\n\nif __name__ == "__main__":\n\n    rng = np.random.default_rng(0)\n    gt = np.zeros((256, 256), np.int32)\n    gt[40:50, 20:200] = 1; gt[120:128, 30:220] = 2\n    prob = (gt > 0).astype(float) * 0.9 + rng.normal(0, 0.12, gt.shape).clip(-0.3, 0.3)\n    prob[40:50, 100:104] = 0.1\n    prob = prob.clip(0, 1)\n    print("default :", evaluate([prob], [gt], **DEFAULTS))\n    print("no close:", evaluate([prob], [gt], **{**DEFAULTS, "close_px": 0}))\n    print("tuned   :", tune([prob], [gt], verbose=False))\n', 'train.py': 'import argparse\nimport hashlib\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport random\nimport shutil\nimport time\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch.utils.data import DataLoader, Dataset\n\nfrom data import load_image, load_instance_mask, random_crop, augment, time_block_split\nfrom losses import seg_loss, distill_loss\nfrom models import build_model, count_params, model_size_mb, predict_prob\nfrom postprocess import prob_to_instances, DEFAULTS\nfrom pq_metric import PQAccumulator\n\n\nclass FilamentDS(Dataset):\n    def __init__(self, df, crop=512, steps=None, sampling="annotation", pos_prob=0.7):\n        self.df, self.crop = df.reset_index(drop=True), crop\n        self.n = len(self.df) if steps is None else steps\n        if self.df.empty or self.n <= 0:\n            raise ValueError("Training data and samples per epoch must be nonempty.")\n        if sampling not in {"annotation", "observation"}:\n            raise ValueError("sampling must be annotation or observation")\n        if not 0 <= pos_prob <= 1:\n            raise ValueError("pos_prob must be between zero and one")\n        self.sampling, self.pos_prob = sampling, pos_prob\n        self.observations = (list(self.df.groupby("obs_id", sort=False).indices.values())\n                             if sampling == "observation" else None)\n\n    def __len__(self):\n        return self.n\n\n    def __getitem__(self, i):\n        if self.observations is None:\n            index = i % len(self.df)\n        else:\n\n            group = self.observations[np.random.randint(len(self.observations))]\n            index = group[np.random.randint(len(group))]\n        row = self.df.iloc[index]\n        img = load_image(row.image_path)\n        mask = (load_instance_mask(row.mask_path) > 0).astype(np.uint8)\n        img, mask = random_crop(img, mask, self.crop, pos_prob=self.pos_prob)\n        img, mask = augment(img, mask)\n        return torch.from_numpy(img)[None], torch.from_numpy(mask.astype(np.float32))[None]\n\n\ndef validate(model, val_df, device, tta=False, params=DEFAULTS,\n             tile_size=512, overlap=64, blend="uniform"):\n\n    if val_df.empty:\n        raise ValueError("Validation subset is empty; check the manifest and split.")\n    acc = PQAccumulator()\n    for image_path, annotations in val_df.groupby("image_path", sort=False):\n        prob = predict_prob(model, load_image(image_path), device, tta=tta,\n                            tile_size=tile_size, overlap=overlap, blend=blend)\n        pred = prob_to_instances(prob, **params)\n        for mask_path in annotations.mask_path:\n            acc.update(load_instance_mask(mask_path), pred)\n    return acc.compute(), None, None\n\n\ndef _seed_worker(_):\n    seed = torch.initial_seed() % (2 ** 32)\n    np.random.seed(seed)\n    random.seed(seed)\n\n\ndef _rng_state(generator):\n    state = np.random.get_state()\n    return {\n        "python": random.getstate(),\n\n        "numpy": [state[0], state[1].tolist(), int(state[2]), int(state[3]), float(state[4])],\n        "torch": torch.get_rng_state(),\n        "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],\n        "loader": generator.get_state(),\n    }\n\n\ndef _restore_rng(state, generator):\n    random.setstate(state["python"])\n    ns = state["numpy"]\n    np.random.set_state((ns[0], np.asarray(ns[1], dtype=np.uint32), ns[2], ns[3], ns[4]))\n    torch.set_rng_state(state["torch"].cpu())\n    generator.set_state(state["loader"].cpu())\n    if torch.cuda.is_available() and state["cuda"]:\n        if len(state["cuda"]) != torch.cuda.device_count():\n            raise ValueError("Resume requires the same CUDA device count for RNG restoration.")\n        torch.cuda.set_rng_state_all([s.cpu() for s in state["cuda"]])\n\n\ndef _save_torch(value, path):\n    temporary = Path(str(path) + ".tmp")\n    torch.save(value, temporary)\n    os.replace(temporary, path)\n\n\ndef _save_json(value, path):\n    temporary = Path(str(path) + ".tmp")\n    with temporary.open("w", encoding="utf-8") as handle:\n        json.dump(value, handle, indent=1)\n    os.replace(temporary, path)\n\n\ndef _check_args(a):\n    if a.epochs <= 0 or a.bs <= 0 or a.val_every <= 0 or a.workers < 0:\n        raise ValueError("epochs, batch size and val-every must be positive; workers cannot be negative.")\n    if a.crop <= 0 or a.crop % 32:\n        raise ValueError("crop must be a positive multiple of 32.")\n    if a.steps_per_epoch is not None and a.steps_per_epoch < a.bs:\n        raise ValueError("samples-per-epoch must be at least the batch size.")\n    if not math.isfinite(a.lr) or a.lr <= 0 or not 0 <= a.pos_prob <= 1:\n        raise ValueError("lr must be positive and finite; pos-prob must be in [0, 1].")\n    if a.val_tile_size <= 0 or not 0 <= a.val_overlap < a.val_tile_size:\n        raise ValueError("Require val-tile-size > 0 and 0 <= val-overlap < val-tile-size.")\n    if a.resume and a.overwrite:\n        raise ValueError("Use --resume or --overwrite, not both.")\n\n\ndef run(a, seed):\n    _check_args(a)\n    torch.manual_seed(seed)\n    np.random.seed(seed)\n    random.seed(seed)\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    out = Path(a.out) / f"seed{seed}"\n    resume_path = (out / "last.pt" if a.resume == "auto" else\n                   Path(a.resume.format(seed=seed)) if a.resume else None)\n    if resume_path is not None and not resume_path.is_file():\n        raise FileNotFoundError(f"Resume checkpoint missing: {resume_path}")\n    if out.exists() and any(out.iterdir()) and not a.resume and not a.overwrite:\n        raise FileExistsError(f"Output directory already contains files: {out}. "\n                              "Use a new --out, --resume, or explicitly --overwrite.")\n    if (a.resume and out.exists() and any(out.iterdir())\n            and resume_path.resolve().parent != out.resolve()):\n        raise FileExistsError("Resuming an external checkpoint requires an empty output directory; "\n                              "use a new --out to protect the existing run.")\n    out.mkdir(parents=True, exist_ok=True)\n    if a.overwrite:\n        for name in ("best.pt", "last.pt", "final.pt", "summary.json"):\n            path = out / name\n            if path.is_file():\n                path.unlink()\n\n    df = pd.read_csv(a.manifest)\n    df = df[df.split == "train"]\n    tr_df, va_df = time_block_split(df, seed=a.split_seed)\n    if tr_df.empty or va_df.empty:\n        raise ValueError("Train or validation subset is empty; check the manifest and split.")\n    print(f"[seed {seed}] train rows {len(tr_df)} | val rows {len(va_df)}")\n    manifest_hash = hashlib.sha256(Path(a.manifest).read_bytes()).hexdigest()\n\n\n    weights = "imagenet" if a.pretrained and not (a.resume or a.init_checkpoint) else None\n    model = build_model(a.model, a.encoder, weights, a.base).to(device)\n    if a.init_checkpoint:\n        initial = torch.load(a.init_checkpoint.format(seed=seed), map_location="cpu", weights_only=True)\n        model.load_state_dict(initial["model"] if "model" in initial else initial, strict=True)\n    teacher = None\n    if a.teacher:\n        teacher = build_model("smp", a.encoder, None).to(device)\n        teacher.load_state_dict(torch.load(a.teacher, map_location="cpu", weights_only=True))\n        teacher.eval()\n\n    generator = torch.Generator().manual_seed(seed)\n    ds = FilamentDS(tr_df, a.crop, steps=a.steps_per_epoch,\n                    sampling=a.sampling, pos_prob=a.pos_prob)\n    dl = DataLoader(ds, batch_size=a.bs, shuffle=True, num_workers=a.workers,\n                    drop_last=True, pin_memory=device.type == "cuda",\n                    worker_init_fn=_seed_worker, generator=generator)\n    if not len(dl):\n        raise ValueError("No complete training batch; reduce --bs or increase --samples-per-epoch.")\n    print(f"Samples per epoch: {len(ds)}; optimizer steps per epoch: {len(dl)}; sampling: {a.sampling}")\n    opt = torch.optim.AdamW(model.parameters(), lr=a.lr, weight_decay=1e-4)\n    total = a.epochs * len(dl)\n    warm = max(1, int(0.05 * total))\n    sched = torch.optim.lr_scheduler.LambdaLR(\n        opt, lambda s: (s + 1) / warm if s < warm else\n        0.5 * (1 + math.cos(math.pi * min(s - warm, total - warm) / max(1, total - warm))))\n    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")\n\n    best, best_epoch, log, start_epoch, elapsed = -1.0, None, [], 0, 0.0\n    if resume_path is not None:\n        saved = torch.load(resume_path, map_location="cpu", weights_only=True)\n        if not isinstance(saved, dict) or saved.get("format_version") != 1:\n            raise ValueError("--resume expects last.pt from this trainer; use --init-checkpoint for weights only.")\n\n        keys = ("model", "encoder", "base", "epochs", "bs", "crop", "lr",\n                "steps_per_epoch", "cldice", "teacher", "kd", "val_every", "workers", "split_seed",\n                "sampling", "pos_prob", "val_tile_size", "val_overlap", "val_blend")\n        differences = [key for key in keys if saved["args"].get(key) != getattr(a, key)]\n        if saved["seed"] != seed or saved["manifest_sha256"] != manifest_hash or differences:\n            raise ValueError(f"Resume configuration/data mismatch (changed options: {differences}). "\n                             "Keep the original settings, including total epochs; "\n                             "use --init-checkpoint for a new fine-tuning run.")\n        if saved["best_val_PQ"] >= 0 and not (out / "best.pt").is_file():\n            previous_best = resume_path.parent / "best.pt"\n            if not previous_best.is_file():\n                raise FileNotFoundError("Resume needs the companion best.pt to preserve the best model.")\n            shutil.copy2(previous_best, out / "best.pt")\n        model.load_state_dict(saved["model"], strict=True)\n        opt.load_state_dict(saved["optimizer"])\n        sched.load_state_dict(saved["scheduler"])\n        scaler.load_state_dict(saved["scaler"])\n        best, best_epoch = saved["best_val_PQ"], saved["best_epoch"]\n        log, start_epoch, elapsed = saved["log"], saved["epoch"] + 1, saved["train_time_s"]\n        _restore_rng(saved["rng"], generator)\n        print(f"Resuming after epoch {start_epoch}; best validation PQ={best:.4f}")\n\n    t0 = time.time()\n    def summary():\n        return dict(seed=seed, best_val_PQ=best, best_epoch=best_epoch,\n                    train_time_s=elapsed + time.time() - t0, params=count_params(model),\n                    size_mb=model_size_mb(model), args=vars(a), log=log,\n                    manifest_sha256=manifest_hash, samples_per_epoch=len(ds),\n                    optimizer_steps_per_epoch=len(dl))\n\n    for ep in range(start_epoch, a.epochs):\n        model.train()\n        run_loss = 0.0\n        for x, y in dl:\n            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)\n            with torch.autocast(device.type, enabled=device.type == "cuda"):\n                logits = model(x)\n                loss = seg_loss(logits, y, w_cldice=a.cldice)\n                if teacher is not None:\n                    with torch.no_grad():\n                        t_logits = teacher(x)\n                    loss = loss + a.kd * distill_loss(logits.float(), t_logits.float())\n            opt.zero_grad(set_to_none=True)\n            scaler.scale(loss).backward()\n            scaler.step(opt)\n            scaler.update()\n            sched.step()\n            run_loss += loss.item()\n\n        msg = dict(epoch=ep, loss=run_loss / len(dl))\n        if (ep + 1) % a.val_every == 0 or ep == a.epochs - 1:\n            res, _, _ = validate(model, va_df, device, tile_size=a.val_tile_size,\n                                  overlap=a.val_overlap, blend=a.val_blend)\n            msg.update({k: float(v) for k, v in res.items()})\n            if res["PQ"] > best:\n                best, best_epoch = float(res["PQ"]), ep\n                _save_torch(model.state_dict(), out / "best.pt")\n        print(msg, flush=True)\n        log.append(msg)\n        report = summary()\n        _save_torch(dict(format_version=1, model=model.state_dict(), optimizer=opt.state_dict(),\n                         scheduler=sched.state_dict(), scaler=scaler.state_dict(), epoch=ep,\n                         seed=seed, best_val_PQ=best, best_epoch=best_epoch, log=log,\n                         train_time_s=report["train_time_s"], args=vars(a),\n                         manifest_sha256=manifest_hash, rng=_rng_state(generator)), out / "last.pt")\n        _save_json(report, out / "summary.json")\n\n    _save_torch(model.state_dict(), out / "final.pt")\n    _save_json(summary(), out / "summary.json")\n    return best\n\n\ndef make_parser():\n    p = argparse.ArgumentParser(description="Train solar filament segmentation with observation-safe validation.")\n    p.add_argument("--manifest", required=True)\n    p.add_argument("--out", required=True)\n    p.add_argument("--model", default="smp", choices=["smp", "light"])\n    p.add_argument("--encoder", default="resnet34")\n    p.add_argument("--base", type=int, default=16)\n    p.add_argument("--pretrained", type=int, default=1)\n    p.add_argument("--epochs", type=int, default=40)\n    p.add_argument("--bs", type=int, default=8)\n    p.add_argument("--crop", type=int, default=512)\n    p.add_argument("--lr", type=float, default=3e-4)\n    p.add_argument("--samples-per-epoch", "--steps-per-epoch", "--steps_per_epoch",\n                   dest="steps_per_epoch", type=int, default=None,\n                   help="Random crop samples per epoch, NOT batches; default one sample per annotation row.")\n    p.add_argument("--sampling", choices=["annotation", "observation"], default="annotation")\n    p.add_argument("--pos-prob", type=float, default=0.7,\n                   help="Probability of anchoring a crop on a foreground pixel.")\n    p.add_argument("--cldice", type=float, default=0.0)\n    p.add_argument("--teacher", default=None)\n    p.add_argument("--kd", type=float, default=1.0)\n    p.add_argument("--val-every", "--val_every", dest="val_every", type=int, default=2)\n    p.add_argument("--val-tile-size", type=int, default=512)\n    p.add_argument("--val-overlap", type=int, default=64)\n    p.add_argument("--val-blend", choices=["uniform", "hann"], default="uniform")\n    p.add_argument("--workers", type=int, default=2)\n    p.add_argument("--seed", type=int, nargs="+", default=[0])\n    p.add_argument("--split-seed", type=int, default=0,\n                   help="Fixed validation split for ALL training seeds; default 0")\n    restarts = p.add_mutually_exclusive_group()\n    restarts.add_argument("--init-checkpoint", default=None,\n                          help="Warm-start weights with a fresh optimizer/schedule; optional {seed} in path.")\n    restarts.add_argument("--resume", nargs="?", const="auto", default=None,\n                          help="Resume last.pt with unchanged training settings; omit path to use --out/seedN/last.pt.")\n    p.add_argument("--overwrite", action="store_true", help="Explicitly replace an existing run\'s generated artifacts.")\n    return p\n\n\nif __name__ == "__main__":\n    a = make_parser().parse_args()\n    scores = [run(a, s) for s in a.seed]\n    print(f"val PQ over seeds: mean={np.mean(scores):.4f}  std={np.std(scores):.4f}")\n', 'infer.py': 'import argparse\nimport hashlib\nimport json\nimport os\nimport tempfile\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\n\nfrom data import load_image, load_instance_mask, time_block_split, rle_counts, rle_encode\nfrom models import build_model, predict_prob, count_params, model_size_mb\nfrom postprocess import prob_to_instances, tune, DEFAULTS\nfrom pq_metric import PQAccumulator\n\n\ndef sha256(path):\n    h = hashlib.sha256()\n    with open(path, "rb") as handle:\n        for part in iter(lambda: handle.read(1024 * 1024), b""):\n            h.update(part)\n    return h.hexdigest()\n\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(path.suffix + ".tmp")\n    tmp.write_text(json.dumps(value, indent=2), encoding="utf-8")\n    os.replace(tmp, path)\n\n\ndef write_submission(rows, path):\n    df = pd.DataFrame(rows, columns=["filament_id", "segmentation_rle"])\n    if df["filament_id"].duplicated().any():\n        raise ValueError("Duplicate filament_id values in Kaggle submission")\n    if df.empty:\n        raise ValueError("No filaments predicted for any test image; review model and thresholds")\n    for item in df.itertuples(index=False):\n        try:\n            counts = rle_counts(item.segmentation_rle)\n        except ValueError as error:\n            raise ValueError(f"Invalid RLE for {item.filament_id}: {error}") from error\n        if sum(counts) != 2048 * 2048 or not sum(counts[1::2]):\n            raise ValueError(f"Invalid RLE for {item.filament_id}: "\n                             "expected a nonempty mask covering exactly 2048 x 2048 pixels")\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    descriptor, name = tempfile.mkstemp(prefix=f".{path.name}.", suffix=".tmp", dir=path.parent)\n    staged = Path(name)\n    try:\n        with os.fdopen(descriptor, "w", encoding="utf-8", newline="") as handle:\n            df.to_csv(handle, index=False, lineterminator="\\n")\n            handle.flush()\n            os.fsync(handle.fileno())\n        restored = pd.read_csv(staged, dtype=str, keep_default_na=False)\n        if not restored.equals(df):\n            raise ValueError("CSV readback differs from the generated predictions")\n        os.replace(staged, path)\n    finally:\n        staged.unlink(missing_ok=True)\n    print(f"Wrote {len(df)} filament predictions to {path}", flush=True)\n\n\ndef validation_partition(val_df, tune_samples, seed=0, validation_samples=0):\n\n    obs = val_df.obs_id.drop_duplicates().to_numpy()\n    if len(obs) < 2:\n        raise ValueError("Need at least two validation observations for calibration and audit")\n    if tune_samples < 1 or validation_samples < 0:\n        raise ValueError("tune_samples must be positive; validation_samples must be nonnegative")\n    obs = np.random.default_rng(seed).permutation(np.sort(obs))\n    n = min(tune_samples, len(obs) - 1)\n    fit_ids, audit_ids = obs[:n], obs[n:]\n    if validation_samples:\n        audit_ids = audit_ids[:validation_samples]\n    return (val_df[val_df.obs_id.isin(fit_ids)].reset_index(drop=True),\n            val_df[val_df.obs_id.isin(audit_ids)].reset_index(drop=True))\n\n\nclass MaskSequence:\n\n    def __init__(self, paths, probabilities=False):\n        self.paths, self.probabilities = list(paths), probabilities\n\n    def __len__(self):\n        return len(self.paths)\n\n    def __getitem__(self, index):\n        path = self.paths[index]\n        return np.load(path, mmap_mode="r") if self.probabilities else load_instance_mask(path)\n\n\ndef predict_ensemble(models, img, device, *, tta=False, tile_size=512,\n                     overlap=64, blend="uniform"):\n\n    if not models:\n        raise ValueError("Need at least one model for probability averaging")\n    result = np.zeros(img.shape, dtype=np.float32)\n    for model in models:\n        result += predict_prob(model, img, device, tta=tta, tile_size=tile_size,\n                               overlap=overlap, blend=blend) / len(models)\n    return result\n\n\ndef cache_predictions(models, rows, device, args, signature):\n    cache_dir = Path(args.cache_dir)\n    cache_dir.mkdir(parents=True, exist_ok=True)\n    paths = {}\n    observations = rows.drop_duplicates("obs_id")\n    for i, row in enumerate(observations.itertuples(), 1):\n        content = dict(inference=signature, image_sha256=sha256(row.image_path))\n        key = hashlib.sha256(json.dumps(content, sort_keys=True).encode()).hexdigest()[:24]\n        path = cache_dir / f"{row.obs_id}_{key}.npy"\n        if not path.is_file():\n            prob = predict_ensemble(models, load_image(row.image_path), device,\n                                    tta=bool(args.tta), tile_size=args.tile_size,\n                                    overlap=args.overlap, blend=args.blend)\n            tmp = path.with_suffix(".npy.tmp")\n            with tmp.open("wb") as handle:\n                np.save(handle, prob)\n            os.replace(tmp, path)\n        paths[row.obs_id] = str(path.resolve())\n        if i % 10 == 0 or i == len(observations):\n            print(f"Validation probability maps: {i}/{len(observations)} (cached maps reused)", flush=True)\n    return paths\n\n\ndef evaluate_rows(rows, paths, params):\n    accumulator, records = PQAccumulator(), []\n    for obs_id, annotations in rows.groupby("obs_id", sort=False):\n        prob = np.load(paths[obs_id], mmap_mode="r")\n        instances = prob_to_instances(prob, **params)\n        for row in annotations.itertuples():\n            gt = load_instance_mask(row.mask_path)\n            accumulator.update(gt, instances)\n            single = PQAccumulator()\n            single.update(gt, instances)\n            records.append(dict(obs_id=obs_id, image_id=row.image_id,\n                                image_path=row.image_path, mask_path=row.mask_path,\n                                prob_path=paths[obs_id], **single.compute()))\n    return accumulator.compute(), records\n\n\ndef main(a):\n    if a.evaluate_only and a.skip_validation:\n        raise ValueError("--evaluate-only needs validation; remove --skip-validation")\n    if a.tune_samples < 1 or a.validation_samples < 0:\n        raise ValueError("Invalid calibration/audit sample counts")\n    if a.use_default_postprocessing and not (a.params and a.skip_tune and a.skip_validation):\n        raise ValueError("--use-default-postprocessing is an export option: also supply "\n                         "--params --skip-tune --skip-validation")\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    checkpoints = [a.ckpt] if isinstance(a.ckpt, str) else a.ckpt\n    if len(set(sha256(p) for p in checkpoints)) != len(checkpoints):\n        raise ValueError("Repeated checkpoint weights would bias the equal-weight ensemble")\n    models = []\n    for checkpoint in checkpoints:\n        model = build_model(a.model, a.encoder, None, a.base).to(device)\n        model.load_state_dict(torch.load(checkpoint, map_location=device, weights_only=True))\n        model.eval()\n        models.append(model)\n        print(f"{checkpoint}: params={count_params(model):,}; size={model_size_mb(model):.2f} MB", flush=True)\n    from importlib.metadata import version\n    signature = dict(checkpoint_sha256=[sha256(p) for p in checkpoints],\n                     model=a.model, encoder=a.encoder, base=a.base, tta=bool(a.tta),\n                     tile_size=a.tile_size, overlap=a.overlap, blend=a.blend,\n                     torch_version=str(torch.__version__),\n                     smp_version=version("segmentation-models-pytorch") if a.model == "smp" else None)\n    df = pd.read_csv(a.manifest, keep_default_na=False)\n    params, calibration = dict(DEFAULTS), {}\n    if a.params:\n        calibration = json.loads(Path(a.params).read_text(encoding="utf-8"))\n        if calibration.get("inference_signature") != signature:\n            raise ValueError("Calibration belongs to different weights/TTA/tiles/blending/library versions. "\n                             "Use matching settings or recalibrate.")\n        if calibration.get("split_seed") != a.split_seed:\n            raise ValueError("Calibration split seed differs from --split-seed")\n        params.update(calibration["postprocessing"])\n        if not a.skip_tune:\n            raise ValueError("Use --skip-tune with --params to reuse a calibration")\n\n    if not a.skip_tune or not a.skip_validation:\n        _, val = time_block_split(df[df.split == "train"], seed=a.split_seed)\n        if val.empty:\n            raise ValueError("Validation subset is empty")\n        if not a.skip_tune:\n            fit, audit = validation_partition(val, a.tune_samples, a.split_seed, a.validation_samples)\n        else:\n            fit, audit = val.iloc[:0], val\n\n            if calibration.get("tuning", {}).get("obs_ids"):\n                audit = audit[~audit.obs_id.isin(calibration["tuning"]["obs_ids"])]\n            if a.validation_samples:\n                selected = audit.obs_id.drop_duplicates().head(a.validation_samples)\n                audit = audit[audit.obs_id.isin(selected)]\n        required = pd.concat([fit, audit if not a.skip_validation else audit.iloc[:0]])\n        paths = cache_predictions(models, required, device, a, signature)\n        if not a.skip_tune:\n            probabilities = MaskSequence([paths[o] for o in fit.obs_id], probabilities=True)\n            gts = MaskSequence(fit.mask_path)\n            params, fit_pq = tune(probabilities, gts, verbose=True)\n            print(f"Calibration fit: {fit.obs_id.nunique()} observations, PQ={fit_pq:.4f}", flush=True)\n            calibration = dict(tuning=dict(PQ=fit_pq, observations=int(fit.obs_id.nunique()),\n                                           annotation_sets=len(fit), obs_ids=fit.obs_id.drop_duplicates().tolist()),\n                               search="two-start coordinate search; not exhaustive")\n        if not a.skip_validation:\n            if audit.empty:\n                raise ValueError("No observations remain for post-processing audit")\n            baseline, _ = evaluate_rows(audit, paths, DEFAULTS)\n            metrics, records = evaluate_rows(audit, paths, params)\n            calibration["validation"] = dict(metrics=metrics, default_postprocessing_metrics=baseline,\n                                             observations=int(audit.obs_id.nunique()), annotation_sets=len(audit),\n                                             obs_ids=audit.obs_id.drop_duplicates().tolist(),\n                                             note="Held out from post-processing tuning. Model checkpoints may "\n                                                  "already have been selected on these observations; this is "\n                                                  "not an untouched model test set or a verified official score.")\n\n\n            pooled = PQAccumulator()\n            fit_metrics = evaluate_rows(fit, paths, params)[0] if not fit.empty else None\n            for result in [metrics, fit_metrics]:\n                if result is not None:\n                    pooled.tp += result["TP"]\n                    pooled.fp += result["FP"]\n                    pooled.fn += result["FN"]\n                    pooled.iou += result["SQ"] * result["TP"]\n                    pooled.split += result["split"]\n                    pooled.merge += result["merge"]\n            all_rows = pd.concat([fit, audit])\n            complete = all_rows.obs_id.nunique() == val.obs_id.nunique()\n            calibration["full_validation" if complete else "sampled_validation"] = dict(\n                metrics=pooled.compute(), observations=int(all_rows.obs_id.nunique()),\n                annotation_sets=len(all_rows),\n                note="Includes observations used for tuning; use the separate post-processing audit for selection.")\n            csv_path = Path(a.params_out or Path(a.out).with_suffix(".calibration.json")).with_suffix(".per_image.csv")\n            csv_path.parent.mkdir(parents=True, exist_ok=True)\n            pd.DataFrame(records).to_csv(csv_path, index=False)\n            print("Audit with default post-processing:", baseline, flush=True)\n            print("Audit with selected post-processing:", metrics, flush=True)\n            if metrics["PQ"] < baseline["PQ"]:\n                print("Tuned post-processing reduced audit PQ. Compare the default recipe before "\n                      "exporting; --use-default-postprocessing can reuse these weights/settings "\n                      "with default instance conversion.", flush=True)\n        calibration.update(postprocessing=params, inference_signature=signature,\n                           checkpoints=checkpoints, split_seed=a.split_seed)\n        params_path = a.params_out or Path(a.out).with_suffix(".calibration.json")\n        write_json(params_path, calibration)\n        print("Saved calibration:", params_path, flush=True)\n    if a.evaluate_only:\n        return calibration\n\n    export_validation = calibration.get("validation")\n    if a.use_default_postprocessing:\n        params = dict(DEFAULTS)\n        if export_validation:\n            export_validation = {**export_validation,\n                                 "metrics": export_validation["default_postprocessing_metrics"]}\n\n    test = df[df.split == "test"]\n    if test.empty or test.obs_id.duplicated().any():\n        raise ValueError("Need a nonempty test set with unique observation IDs")\n    rows, missing = [], 0\n    start = time.perf_counter()\n    for index, row in enumerate(test.itertuples(), 1):\n        prob = predict_ensemble(models, load_image(row.image_path), device,\n                                tta=bool(a.tta), tile_size=a.tile_size,\n                                overlap=a.overlap, blend=a.blend)\n        inst = prob_to_instances(prob, **params)\n        if inst.shape != (2048, 2048):\n            raise ValueError(f"Expected 2048 x 2048 prediction for {row.obs_id}, got {inst.shape}")\n        ids = np.unique(inst)\n        ids = ids[ids > 0]\n        missing += int(len(ids) == 0)\n        for number, instance_id in enumerate(ids, 1):\n            rows.append((f"{row.obs_id}_{number}", rle_encode(inst == instance_id)))\n        if index % 30 == 0 or index == len(test):\n            print(f"Predicted {index}/{len(test)} test images", flush=True)\n    if device.type == "cuda":\n        torch.cuda.synchronize()\n    elapsed = time.perf_counter() - start\n    write_submission(rows, a.out)\n    details = dict(checkpoints=checkpoints, inference_signature=signature, csv_sha256=sha256(a.out),\n                   model=a.model, encoder=a.encoder, output=a.out, postprocessing=params,\n                   test_images=len(test), predicted_filaments=len(rows),\n                   images_without_predictions=missing, seconds_per_image=elapsed / len(test),\n                   tta=bool(a.tta), split_seed=a.split_seed, calibration_file=a.params,\n                   used_tuned_postprocessing=bool((a.params or not a.skip_tune)\n                                                 and not a.use_default_postprocessing),\n                   validation=export_validation)\n    write_json(Path(a.out).with_suffix(".metadata.json"), details)\n    print(f"End-to-end seconds/test image: {elapsed / len(test):.3f}; empty images: {missing}", flush=True)\n    return details\n\n\ndef parser():\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument("--manifest", required=True)\n    p.add_argument("--ckpt", required=True, nargs="+", help="One checkpoint, or several compatible models for a probability ensemble")\n    p.add_argument("--out", default="submission.csv")\n    p.add_argument("--model", default="smp", choices=["smp", "light"])\n    p.add_argument("--encoder", default="resnet34")\n    p.add_argument("--base", type=int, default=16)\n    p.add_argument("--seed", type=int, default=0, help="Legacy option; use --split-seed for the validation split")\n    p.add_argument("--tta", type=int, choices=[0, 1], default=0)\n    p.add_argument("--tile-size", type=int, default=512)\n    p.add_argument("--overlap", type=int, default=64)\n    p.add_argument("--blend", choices=["uniform", "hann"], default="uniform")\n    p.add_argument("--skip-tune", action="store_true")\n    p.add_argument("--skip-validation", action="store_true")\n    p.add_argument("--evaluate-only", action="store_true", help="Calibrate/evaluate without predicting test images")\n    p.add_argument("--tune-samples", type=int, default=24, help="Unique validation observations used to tune post-processing")\n    p.add_argument("--validation-samples", type=int, default=0, help="Limit audit observations for a smoke check; 0 audits all remaining observations")\n    p.add_argument("--cache-dir", default="runs/prob_cache", help="Disk cache of validation probability maps (about 16 MB per observation/configuration)")\n    p.add_argument("--params", help="Reuse calibration JSON, requiring matching inference settings")\n    p.add_argument("--params-out", help="Where to save calibration and diagnostics")\n    p.add_argument("--use-default-postprocessing", action="store_true",\n                   help="Export with DEFAULTS when its audit beats tuning; requires --params --skip-tune --skip-validation")\n    p.add_argument("--split-seed", type=int, default=0, help="Must match training")\n    return p\n\n\nif __name__ == "__main__":\n    main(parser().parse_args())\n', 'submission_check.py': '\n\nimport argparse\nimport hashlib\nimport re\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom data import rle_counts, rle_decode\n\n\nEXPECTED_COLUMNS = ["filament_id", "segmentation_rle"]\n\n\ndef _validate_counts(encoded, pixels):\n    counts = rle_counts(encoded)\n    if sum(counts) != pixels:\n        raise ValueError(f"RLE runs cover {sum(counts)} pixels; expected {pixels}")\n    return counts\n\n\ndef validate_submission(frame, test_ids=None, shape=(2048, 2048)):\n    if len(shape) != 2 or any(not isinstance(n, (int, np.integer)) or n < 1 for n in shape):\n        raise ValueError("Mask shape must contain two positive integer dimensions")\n    if list(frame.columns) != EXPECTED_COLUMNS:\n        raise ValueError(f"Unexpected columns: {list(frame.columns)}. Expected {EXPECTED_COLUMNS}")\n    if frame.empty:\n        raise ValueError("Submission has no filament predictions")\n    if frame.filament_id.duplicated().any():\n        raise ValueError("Duplicate filament_id detected")\n    if not frame.segmentation_rle.map(lambda value: isinstance(value, str) and bool(value)).all():\n        raise ValueError("Submission contains empty RLE strings")\n    if not frame.filament_id.map(lambda value: isinstance(value, str) and bool(value)).all():\n        raise ValueError("Submission contains empty filament IDs")\n\n    test_ids = None if test_ids is None else set(map(str, test_ids))\n    unknown = []\n    groups = {}\n    for filament_id, encoded in frame.itertuples(index=False, name=None):\n        found = re.fullmatch(r"(.+)_(\\d+)", filament_id)\n        if found is None or (test_ids is not None and found.group(1) not in test_ids):\n            unknown.append(filament_id)\n        else:\n            groups.setdefault(found.group(1), []).append((filament_id, encoded))\n    if unknown:\n        raise ValueError(f"Found unknown or malformed filament IDs, examples: {unknown[:3]}")\n\n    pixels = int(shape[0]) * int(shape[1])\n    for obs_id, items in groups.items():\n        owners = np.zeros(shape, dtype=np.int32)\n        for number, (filament_id, encoded) in enumerate(items, 1):\n            try:\n                _validate_counts(encoded, pixels)\n                mask = np.asarray(rle_decode(encoded, shape))\n            except Exception as error:\n                raise ValueError(f"Invalid RLE for {filament_id}: {error}") from error\n            if mask.shape != tuple(shape) or not np.isin(mask, [0, 1]).all() or not mask.any():\n                raise ValueError(f"Invalid or empty decoded mask for {filament_id}")\n            foreground = mask.astype(bool)\n            overlaps = foreground & (owners > 0)\n            if overlaps.any():\n                conflicting = [items[i - 1][0] for i in np.unique(owners[overlaps])]\n                raise ValueError(f"Overlapping masks in {obs_id}: {filament_id} overlaps "\n                                 f"{conflicting} at {np.count_nonzero(overlaps)} pixels")\n            owners[foreground] = number\n    missing = [] if test_ids is None else sorted(test_ids - groups.keys())\n    return dict(filament_rows=len(frame), represented_images=len(groups),\n                test_images=None if test_ids is None else len(test_ids),\n                missing_images=missing, decoded_masks=len(frame))\n\n\ndef check_file(csv_path, manifest_path, samples=5):\n    path = Path(csv_path)\n    if not path.is_file():\n        raise FileNotFoundError(path)\n    digest = hashlib.sha256()\n    with path.open("rb") as handle:\n        for part in iter(lambda: handle.read(1024 * 1024), b""):\n            digest.update(part)\n    csv_sha256 = digest.hexdigest()\n    print(f"CSV SHA256: {csv_sha256}")\n    frame = pd.read_csv(path, dtype=str, keep_default_na=False)\n    manifest = pd.read_csv(manifest_path, dtype={"obs_id": str}, keep_default_na=False)\n    test_ids = set(manifest.loc[manifest.split == "test", "obs_id"].astype(str))\n    result = validate_submission(frame, test_ids)\n    result["csv_sha256"] = csv_sha256\n    print(f"LOCAL FORMAT CHECK PASSED: {len(frame)} filament rows, "\n          f"{result[\'represented_images\']}/{len(test_ids)} test images represented")\n    print(f"Decoded every RLE mask and checked all {len(frame)} masks for overlaps")\n    if result["missing_images"]:\n        print(f"Test images without predictions: {result[\'missing_images\']}")\n    print("Competition acceptance is checked separately by Kaggle; this check does not guarantee acceptance.")\n    return result\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--csv", required=True)\n    parser.add_argument("--manifest", default="manifest.csv")\n    parser.add_argument("--sample", type=int, default=5)\n    arguments = parser.parse_args()\n    check_file(arguments.csv, arguments.manifest, arguments.sample)\n', 'compress.py': '\n\n\nimport argparse, copy\nimport pandas as pd\nimport torch\nimport torch.nn as nn\nimport torch.nn.utils.prune as prune\nfrom torch.utils.data import DataLoader\n\nfrom data import time_block_split\nfrom losses import seg_loss\nfrom models import build_model, count_params, model_size_mb, time_inference\nfrom train import FilamentDS, validate\n\n\ndef conv_params(model):\n    return [(m, "weight") for m in model.modules() if isinstance(m, nn.Conv2d)]\n\n\ndef sparsity(model):\n    w = torch.cat([m.weight.flatten() for m, _ in conv_params(model)])\n    return float((w == 0).float().mean())\n\n\ndef prune_model(model, amount, tr_df, device, finetune_epochs=0, crop=256, bs=1, workers=0):\n    m = copy.deepcopy(model)\n    prune.global_unstructured(conv_params(m), pruning_method=prune.L1Unstructured, amount=amount)\n    if finetune_epochs:\n        dl = DataLoader(FilamentDS(tr_df, crop, steps=200), batch_size=bs, shuffle=True, num_workers=workers)\n        opt = torch.optim.AdamW(m.parameters(), lr=1e-4)\n        scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")\n        for _ in range(finetune_epochs):\n            m.train()\n            for x, y in dl:\n                x, y = x.to(device), y.to(device)\n                with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):\n                    loss = seg_loss(m(x), y)\n                opt.zero_grad(set_to_none=True)\n                scaler.scale(loss).backward()\n                scaler.step(opt)\n                scaler.update()\n    for mod, name in conv_params(m):\n        prune.remove(mod, name)\n    return m\n\n\ndef evaluate_variant(name, model, va_df, device, half=False):\n    m = copy.deepcopy(model).half() if half else model\n    t = time_inference(m, device, shape=(512, 512), half=half)\n\n    from models import predict_prob\n    from data import load_image, load_instance_mask\n    from postprocess import prob_to_instances, DEFAULTS\n    from pq_metric import PQAccumulator\n    acc = PQAccumulator()\n    for _, r in va_df.iterrows():\n        prob = predict_prob(m, load_image(r.image_path), device, half=half)\n        acc.update(load_instance_mask(r.mask_path), prob_to_instances(prob, **DEFAULTS))\n    pq = acc.compute()["PQ"]\n    return dict(variant=name, params=count_params(model), sparsity=round(sparsity(model), 3),\n                size_mb=round(model_size_mb(m), 2), ms_per_512tile_net=round(t * 1000, 1), val_PQ=round(float(pq), 4))\n\n\nif __name__ == "__main__":\n    p = argparse.ArgumentParser()\n    p.add_argument("--manifest", required=True); p.add_argument("--ckpt", required=True)\n    p.add_argument("--model", default="smp"); p.add_argument("--encoder", default="resnet34")\n    p.add_argument("--base", type=int, default=16); p.add_argument("--finetune", type=int, default=2)\n    p.add_argument("--bs", type=int, default=1); p.add_argument("--crop", type=int, default=256)\n    p.add_argument("--workers", type=int, default=0); p.add_argument("--split-seed", type=int, default=0)\n    p.add_argument("--val-limit", type=int, default=24, help="0 evaluates the complete validation set")\n    p.add_argument("--out", default="compression_results.csv")\n    a = p.parse_args()\n\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    df = pd.read_csv(a.manifest); tr, va = time_block_split(df[df.split == "train"], seed=a.split_seed)\n    if a.val_limit > 0:\n        va = va.sample(n=min(a.val_limit, len(va)), random_state=a.split_seed).reset_index(drop=True)\n    print(f"Compression evaluation uses {len(va)} validation annotation sets")\n    base = build_model(a.model, a.encoder, None, a.base).to(device)\n    base.load_state_dict(torch.load(a.ckpt, map_location=device, weights_only=True))\n\n    rows = [evaluate_variant("fp32 baseline", base, va, device)]\n    if device.type == "cuda":\n        rows.append(evaluate_variant("fp16", base, va, device, half=True))\n    for amt in [0.3, 0.5, 0.7, 0.9]:\n        pm = prune_model(base, amt, tr, device, a.finetune, crop=a.crop, bs=a.bs, workers=a.workers)\n        rows.append(evaluate_variant(f"prune {int(amt*100)}% (+{a.finetune} ep finetune)", pm, va, device))\n        del pm\n        if device.type == "cuda":\n            torch.cuda.empty_cache()\n    print("NOTE: Unstructured zero weights remain in dense tensors; file size and inference speed may not improve.")\n    print("Network-only timing is NOT end-to-end timing; infer.py measures full pipeline.")\n    print(pd.DataFrame(rows).to_string(index=False))\n    pd.DataFrame(rows).to_csv(a.out, index=False)\n', 'timing_probe.py': '\n\n\nimport argparse\nimport math\nimport time\nfrom contextlib import nullcontext\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch.utils.data import DataLoader\n\nfrom data import load_image, load_instance_mask, time_block_split\nfrom losses import seg_loss\nfrom models import build_model, predict_prob\nfrom postprocess import prob_to_instances, DEFAULTS\nfrom pq_metric import PQAccumulator\nfrom train import FilamentDS\n\n\ndef synchronize(device):\n    if device.type == "cuda":\n        torch.cuda.synchronize()\n\n\ndef main(args):\n    df = pd.read_csv(args.manifest)\n    df = df[df.split == "train"]\n    tr, va = time_block_split(df, seed=args.split_seed)\n    if tr.empty or va.empty:\n        raise ValueError("Train or validation subset empty; check manifest and split.")\n    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    print(f"Device={dev}; train annotation sets={len(tr)}, validation sets={len(va)}")\n    if dev.type == "cuda":\n        print("GPU:", torch.cuda.get_device_name(0))\n    net = build_model(args.model, args.encoder,\n                      "imagenet" if args.pretrained else None, args.base).to(dev)\n    opt = torch.optim.AdamW(net.parameters(), lr=3e-4)\n    scaler = torch.amp.GradScaler("cuda", enabled=dev.type == "cuda")\n    ds = FilamentDS(tr, crop=args.crop, steps=(args.batches + 1) * args.bs)\n    dl = DataLoader(ds, batch_size=args.bs, shuffle=False, num_workers=0,\n                    drop_last=True)\n    batch_seconds = []\n    net.train()\n    for i, (x, y) in enumerate(dl):\n        if i >= args.batches + 1:\n            break\n        synchronize(dev)\n        t0 = time.perf_counter()\n        x, y = x.to(dev), y.to(dev)\n        with torch.autocast(device_type=dev.type, enabled=dev.type == "cuda"):\n            logits = net(x)\n            loss = seg_loss(logits, y, w_cldice=args.cldice)\n        opt.zero_grad(set_to_none=True)\n        scaler.scale(loss).backward()\n        scaler.step(opt)\n        scaler.update()\n        synchronize(dev)\n        if i > 0:\n            batch_seconds.append(time.perf_counter() - t0)\n    avg_batch = np.mean(batch_seconds)\n    train_batches = len(tr) // args.bs\n    train_epoch = train_batches * avg_batch\n    print(f"Measured training batch (excluding disk-to-batch DataLoader latency): {avg_batch:.3f} s")\n    print(f"Projected train-only epoch: ~{train_epoch/60:.1f} minutes, "\n          "BEFORE data-loading / validation / checkpoint overhead")\n\n    net.eval()\n    results = []\n    acc = PQAccumulator()\n    for _, row in va.head(args.val_images).iterrows():\n        synchronize(dev)\n        t0 = time.perf_counter()\n        img = load_image(row.image_path)\n        prob = predict_prob(net, img, dev, tta=bool(args.tta))\n        inst = prob_to_instances(prob, **DEFAULTS)\n        gt = load_instance_mask(row.mask_path)\n        acc.update(gt, inst)\n        synchronize(dev)\n        dt = time.perf_counter() - t0\n        results.append(dt)\n        print(f"Validation image {len(results)}: {dt:.1f} s full pipeline")\n    val_per_image = np.mean(results)\n    validation_pass = val_per_image * len(va)\n    n_validation_passes = math.ceil(args.epochs / args.val_every)\n    projection = args.epochs * train_epoch + n_validation_passes * validation_pass\n    print(f"Projected validation pass: ~{validation_pass/60:.1f} minutes")\n    print(f"Projected {args.epochs}-epoch run with validation every {args.val_every}: "\n          f"~{projection/3600:.1f} hours minimum estimate")\n    print("This estimate is indicative only: excludes data-loader stall, model saving, "\n          "post-processing search, compression and Kaggle submission time. "\n          "Validation predictions from this probe are from an almost-untrained model "\n          "and must NOT be reported as assignment results.")\n\n\nif __name__ == "__main__":\n    p = argparse.ArgumentParser(description="Hardware-based COS711 A2 runtime probe")\n    p.add_argument("--manifest", default="manifest.csv")\n    p.add_argument("--model", choices=["light", "smp"], default="light")\n    p.add_argument("--encoder", default="resnet34")\n    p.add_argument("--pretrained", type=int, default=0)\n    p.add_argument("--base", type=int, default=16)\n    p.add_argument("--bs", type=int, default=2)\n    p.add_argument("--crop", type=int, default=256)\n    p.add_argument("--batches", type=int, default=4)\n    p.add_argument("--val-images", type=int, default=2)\n    p.add_argument("--tta", type=int, default=0)\n    p.add_argument("--cldice", type=float, default=0.0)\n    p.add_argument("--epochs", type=int, default=40)\n    p.add_argument("--val-every", type=int, default=5)\n    p.add_argument("--split-seed", type=int, default=0)\n    main(p.parse_args())\n'}
for filename, source in EMBEDDED_SOURCES.items():
    compile(source, filename, 'exec')
    (CODE_DIR / filename).write_text(source, encoding='utf-8')
sys.path.insert(0, str(CODE_DIR))
for filename in EMBEDDED_SOURCES:
    sys.modules.pop(Path(filename).stem, None)
print('Embedded modules:', ', '.join(EMBEDDED_SOURCES))

In [ ]:
if DATA_ROOT:
    candidates = [Path(DATA_ROOT).expanduser().resolve()]
elif ON_KAGGLE:
    candidates = sorted({p.parent.parent for p in Path("/kaggle/input").rglob("train_images")
                         if p.parent.name == "train"})
else:
    candidates = [ORIGINAL_CWD / "MAGFiLO_1.0_Kaggle_2026"]
candidates = [p for p in candidates if (p / "train" / "train_images").is_dir()
              and (p / "test" / "test_images").is_dir()
              and len(list((p / "train").glob("*.json"))) == 1]
if len(candidates) != 1:
    raise ValueError(f"Set DATA_ROOT to the official competition data root. Candidates: {candidates}")
COMPETITION_ROOT = candidates[0]
MANIFEST = WORK / "official_manifest.csv"
MASK_CACHE = WORK / "official_masks"
subprocess.run([sys.executable, "-u", str(CODE_DIR / "data.py"), "build",
                "--root", str(COMPETITION_ROOT), "--cache", str(MASK_CACHE),
                "--out", str(MANIFEST)], check=True, cwd=str(CODE_DIR))
frame = pd.read_csv(MANIFEST, keep_default_na=False)
n_test = int((frame.split == "test").sum())
if n_test != EXPECTED_TEST_IMAGES:
    raise ValueError(f"Expected {EXPECTED_TEST_IMAGES} official test images; found {n_test}")
from data import time_block_split
train_rows, val_rows = time_block_split(frame[frame.split == "train"], seed=SPLIT_SEED)
assert set(train_rows.obs_id).isdisjoint(val_rows.obs_id)
print("Training:", len(train_rows), "annotation sets /", train_rows.obs_id.nunique(), "observations")
print("Validation:", len(val_rows), "annotation sets /", val_rows.obs_id.nunique(), "observations")
print("The same observation's annotator versions remain together in the split.")

In [ ]:
from checkpoint_io import find_checkpoints, prepare_checkpoint

ACTIVE_CHECKPOINTS = []
if RUN_MODE not in {"train", "all"}:
    if CHECKPOINTS:
        ACTIVE_CHECKPOINTS = [Path(p).expanduser().resolve() for p in CHECKPOINTS]
    elif RUN_MODE == "submit":
        roots = [Path("/kaggle/input")] if ON_KAGGLE else [ORIGINAL_CWD / "runs"]
        found = find_checkpoints(roots)
        if len(found) != 1:
            raise ValueError(
                f"Found {len(found)} best.pt files: {found}. Attach the extracted weights "
                "as a Dataset input, or set CHECKPOINTS explicitly to your intended model."
            )
        ACTIVE_CHECKPOINTS = found
    else:
        search_roots = [Path("/kaggle/input")] if ON_KAGGLE else [ORIGINAL_CWD / "runs"]
        for seed in CHECKPOINT_SEEDS:
            found = sorted({p.resolve() for p in find_checkpoints(search_roots)
                            if p.parent.name == f"seed{seed}" and p.parent.parent.name == "base"})
            if len(found) != 1:
                raise ValueError(
                    f"RUN_MODE={RUN_MODE!r} needs trained weights. Found {len(found)} baseline "
                    f"seed{seed} checkpoints: {found}. Set CHECKPOINTS explicitly to an existing best.pt. "
                    "For a fresh model, set RUN_MODE='train' and choose TRAIN_SEEDS, "
                    "then rerun the notebook from the top."
                )
            ACTIVE_CHECKPOINTS.append(found[0])
    if not ACTIVE_CHECKPOINTS or len(set(ACTIVE_CHECKPOINTS)) != len(ACTIVE_CHECKPOINTS):
        raise ValueError("Supply at least one unique checkpoint")
    for index, checkpoint in enumerate(ACTIVE_CHECKPOINTS):
        if checkpoint.suffix.lower() == ".zip":
            raise ValueError("Extract the downloaded ZIP and select best.pt inside it, not the outer archive.")
        if checkpoint.is_dir():
            restored = prepare_checkpoint(checkpoint, WORK / "checkpoint_cache")
            print("Restored unpacked checkpoint:", checkpoint, "->", restored)
            ACTIVE_CHECKPOINTS[index] = checkpoint = restored
        if not checkpoint.is_file():
            roots = [WORK, Path("/kaggle/input")] if ON_KAGGLE else [WORK, ORIGINAL_CWD / "runs"]
            available = sorted({p.resolve() for root in roots if root.is_dir()
                                for p in root.rglob("best.pt") if p.is_file()})
            listing = "\n".join(map(str, available)) or "None found in the current working files or attached inputs."
            raise FileNotFoundError(
                f"Checkpoint not found: {checkpoint}\nAvailable best.pt files:\n{listing}\n"
                "Set CHECKPOINTS to the exact path of your intended trained model. "
                "In a new Kaggle session, attach the saved training outputs and use their input path. "
                "If no weights were saved, finish or resume training first; calibration does not create them."
            )
        with checkpoint.open("rb") as handle:
            if handle.read(64).startswith(b"version https://git-lfs.github.com/spec"):
                raise ValueError(f"Git LFS pointer found instead of model weights: {checkpoint}")
    print("Selected checkpoints:", *ACTIVE_CHECKPOINTS, sep="\n  ")

if CALIBRATION_PATH and RUN_MODE in {"calibrate", "all", "submit"}:
    raise ValueError("CALIBRATION_PATH reuses an existing report in infer mode; leave blank to calibrate.")
CALIBRATION = (Path(CALIBRATION_PATH).expanduser().resolve()
               if CALIBRATION_PATH and RUN_MODE == "infer" else EXP_DIR / "calibration.json")
OUTPUT_CSV = EXP_DIR / "submission.csv"
TRAIN_OUT = WORK / "trained_models" / EXPERIMENT

def run_script(name, arguments):
    command = [sys.executable, "-u", str(CODE_DIR / name), *map(str, arguments)]
    print("Running:", " ".join(command), flush=True)
    subprocess.run(command, check=True, cwd=str(CODE_DIR))

def inference_arguments():
    return ["--manifest", MANIFEST, "--ckpt", *ACTIVE_CHECKPOINTS,
            "--model", MODEL, "--encoder", ENCODER, "--base", BASE,
            "--split-seed", SPLIT_SEED, "--tta", int(TTA), "--tile-size", TILE_SIZE,
            "--overlap", TILE_OVERLAP, "--blend", BLEND,
            "--cache-dir", WORK / "prob_cache"]

## Training

Modes `all` and `train` train the selected seeds. In `all`, their verified best.pt files are passed directly to calibration and prediction. Use `submit` to reuse existing trained weights.

In [ ]:
if RUN_MODE in {"train", "all"}:
    if INIT_CHECKPOINT and RESUME_CHECKPOINT:
        raise ValueError("Choose INIT_CHECKPOINT or RESUME_CHECKPOINT")
    if RESUME_CHECKPOINT and len(TRAIN_SEEDS) != 1:
        raise ValueError("Resume one seed at a time")
    args = ["--manifest", MANIFEST, "--out", TRAIN_OUT, "--model", MODEL,
            "--encoder", ENCODER, "--base", BASE, "--epochs", EPOCHS,
            "--samples-per-epoch", SAMPLES_PER_EPOCH, "--bs", BATCH_SIZE,
            "--crop", CROP_SIZE, "--lr", LEARNING_RATE, "--cldice", CLDICE_WEIGHT,
            "--val-every", 5, "--workers", WORKERS, "--sampling", SAMPLING,
            "--split-seed", SPLIT_SEED, "--seed", *TRAIN_SEEDS]
    if INIT_CHECKPOINT:
        args += ["--init-checkpoint", Path(INIT_CHECKPOINT).expanduser().resolve(), "--pretrained", 0]
    if RESUME_CHECKPOINT:
        args += ["--resume", Path(RESUME_CHECKPOINT).expanduser().resolve()]
    run_script("train.py", args)
    trained_checkpoints = [TRAIN_OUT / f"seed{seed}" / "best.pt" for seed in TRAIN_SEEDS]
    missing_checkpoints = [str(path) for path in trained_checkpoints if not path.is_file()]
    if missing_checkpoints:
        raise RuntimeError(
            f"Training returned without the expected best.pt files: {missing_checkpoints}. "
            "Check the training output before switching to calibration."
        )
    if RUN_MODE == "all":
        ACTIVE_CHECKPOINTS = trained_checkpoints
        print("Training complete. Continuing with calibration:", *ACTIVE_CHECKPOINTS, sep="\n  ")
    else:
        print("Next: set RUN_MODE='calibrate' and CHECKPOINTS to:")
        print([str(path) for path in trained_checkpoints])
    print("Save the complete seed folders before ending this session or starting a new execution.")

## Calibration

Modes `all` and `submit` calibrate automatically and select tuned or default post-processing using audit PQ. Mode `calibrate` runs this stage separately.

In [ ]:
if RUN_MODE in {"calibrate", "all", "submit"}:
    run_script("infer.py", inference_arguments() + [
        "--evaluate-only", "--tune-samples", TUNE_OBSERVATIONS, "--params-out", CALIBRATION])
    if RUN_MODE == "calibrate":
        print("Calibration complete. Next set RUN_MODE='infer' with the same checkpoint/settings "
              "to generate test predictions at:", OUTPUT_CSV)
if RUN_MODE in {"calibrate", "infer", "all", "submit"} and CALIBRATION.is_file():
    calibration_report = json.loads(CALIBRATION.read_text(encoding="utf-8"))
    for field in ("postprocessing", "tuning", "validation", "full_validation", "sampled_validation"):
        if field in calibration_report:
            print(field + ":", json.dumps(calibration_report[field], indent=2))
    audit = calibration_report.get("validation", {})
    tuned_pq = audit.get("metrics", {}).get("PQ")
    default_pq = audit.get("default_postprocessing_metrics", {}).get("PQ")
    if RUN_MODE in {"all", "submit"}:
        if tuned_pq is None or default_pq is None:
            raise RuntimeError("Automatic export requires the calibration audit comparison.")
        USE_TUNED_POSTPROCESSING = tuned_pq >= default_pq
        print("Selected post-processing:", "tuned" if USE_TUNED_POSTPROCESSING else "default",
              "| tuned audit PQ:", tuned_pq, "| default audit PQ:", default_pq)
    elif tuned_pq is not None and default_pq is not None and tuned_pq < default_pq:
        print("Default post-processing scored higher on the audit observations. "
              "Set USE_TUNED_POSTPROCESSING=False when exporting this recipe, "
              "or improve calibration before choosing a submission.")

## Submission

Modes `all` and `submit` export automatically. Upload the final submission.csv from the working root to the competition. Mode `infer` exports separately using an existing calibration report.

In [ ]:
if RUN_MODE in {"infer", "all", "submit"}:
    if not CALIBRATION.is_file():
        raise FileNotFoundError(f"Run calibration first with these settings: {CALIBRATION}")
    export_args = inference_arguments() + ["--params", CALIBRATION,
                  "--skip-tune", "--skip-validation", "--out", OUTPUT_CSV]
    if not USE_TUNED_POSTPROCESSING:
        export_args += ["--use-default-postprocessing"]
    run_script("infer.py", export_args)
    run_script("submission_check.py", ["--csv", OUTPUT_CSV, "--manifest", MANIFEST])
    print("Upload this CSV:", OUTPUT_CSV)
    if RUN_MODE in {"all", "submit"}:
        source_metadata = OUTPUT_CSV.with_suffix(".metadata.json")
        if not OUTPUT_CSV.is_file() or not source_metadata.is_file():
            raise RuntimeError("Inference returned without the expected CSV and metadata files.")
        root_csv = WORK / "submission.csv"
        root_metadata = WORK / "submission.metadata.json"
        import tempfile
        with tempfile.NamedTemporaryFile(dir=WORK, prefix=".submission.", suffix=".csv", delete=False) as handle:
            staged_csv = Path(handle.name)
        try:
            shutil.copyfile(OUTPUT_CSV, staged_csv)
            if staged_csv.read_bytes() != OUTPUT_CSV.read_bytes():
                raise RuntimeError("Final CSV copy differs from the checked predictions")
            staged_csv.replace(root_csv)
        finally:
            staged_csv.unlink(missing_ok=True)
        export_metadata = json.loads(source_metadata.read_text(encoding="utf-8"))
        export_metadata.update(output=str(root_csv), experiment_output=str(OUTPUT_CSV))
        root_metadata.write_text(json.dumps(export_metadata, indent=2), encoding="utf-8")
        summary = dict(mode=RUN_MODE, checkpoints=[str(p) for p in ACTIVE_CHECKPOINTS],
                       calibration=str(CALIBRATION), output=str(root_csv), metadata=str(root_metadata),
                       experiment_output=str(OUTPUT_CSV),
                       used_tuned_postprocessing=USE_TUNED_POSTPROCESSING,
                       validation=export_metadata.get("validation"))
        (EXP_DIR / "run_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
        print("Pipeline complete. Upload this final CSV to the competition:", root_csv)
        if RUN_COMPRESSION:
            run_script("compress.py", ["--manifest", MANIFEST, "--ckpt", ACTIVE_CHECKPOINTS[0],
                       "--model", MODEL, "--encoder", ENCODER, "--base", BASE,
                       "--split-seed", SPLIT_SEED, "--finetune", 0, "--val-limit", 24,
                       "--out", EXP_DIR / "compression_results.csv"])

In [ ]:
for summary_path in sorted(TRAIN_OUT.glob("seed*/summary.json")):
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    history = pd.DataFrame(summary["log"])
    fig, axes = plt.subplots(1, 2, figsize=(11, 3))
    axes[0].plot(history.epoch + 1, history.loss)
    axes[0].set(xlabel="Epoch", ylabel="Training loss", title=summary_path.parent.name)
    if "PQ" in history:
        evaluated = history.dropna(subset=["PQ"])
        axes[1].plot(evaluated.epoch + 1, evaluated.PQ, "o-")
    axes[1].set(xlabel="Epoch", ylabel="Default-postprocessing validation PQ")
    fig.tight_layout(); plt.show()
    print("Best validation PQ:", summary["best_val_PQ"], "| Training seconds:", summary["train_time_s"])

In [ ]:
per_image_csv = CALIBRATION.with_suffix(".per_image.csv")
if RUN_MODE in {"calibrate", "all", "submit"} and per_image_csv.is_file():
    from data import load_image, load_instance_mask
    from postprocess import prob_to_instances
    from pq_metric import PQAccumulator
    scores = pd.read_csv(per_image_csv)
    display(scores.sort_values("PQ").head(8))
    chosen = pd.concat([scores.sort_values("PQ").drop_duplicates("obs_id").head(2),
                        scores.sort_values("PQ", ascending=False).drop_duplicates("obs_id").head(2)])
    chosen = chosen.drop_duplicates(["obs_id", "mask_path"])
    from postprocess import DEFAULTS
    params = calibration_report["postprocessing"] if USE_TUNED_POSTPROCESSING else dict(DEFAULTS)
    fig, axes = plt.subplots(len(chosen), 3, figsize=(14, 4 * len(chosen)), squeeze=False)
    for axis, row in zip(axes, chosen.itertuples(index=False)):
        image = load_image(row.image_path)
        gt = load_instance_mask(row.mask_path)
        probability = np.load(row.prob_path)
        pred = prob_to_instances(probability, **params)
        displayed_score = PQAccumulator()
        displayed_score.update(gt, pred)
        axis[0].imshow(image, cmap="gray"); axis[0].set_title(row.obs_id)
        axis[1].imshow(image, cmap="gray")
        axis[1].imshow(np.ma.masked_where(gt == 0, gt), cmap="tab20", alpha=.65)
        axis[1].set_title("One annotator's instances")
        axis[2].imshow(image, cmap="gray")
        axis[2].imshow(np.ma.masked_where(pred == 0, pred), cmap="tab20", alpha=.65)
        axis[2].set_title(f"Prediction; local PQ={displayed_score.compute()['PQ']:.3f}")
        for item in axis:
            item.axis("off")
    fig.tight_layout()
    fig.savefig(EXP_DIR / "validation_examples.png", dpi=120)
    plt.show()